# 🚀 t4-diffusion: TensorRT-Optimized Stable Diffusion on Free Colab T4

This notebook demonstrates running Stable Diffusion on Google Colab's free T4 GPU with TensorRT INT8 optimizations.

**Features:**
- INT8 quantization via nvidia-modelopt (1.5-2x speedup)
- TensorRT compilation for optimized inference
- Feature caching for additional acceleration
- VRAM monitoring with 15.6GB T4 limit enforcement

**Requirements:**
- Google Colab with T4 GPU runtime
- PyTorch 2.7.x (Colab default, built against CUDA 12.6)
- ~10GB free VRAM

## 1. Setup & Installation

Install torch-tensorrt 2.7.0 (matching Colab's PyTorch 2.7.x) and nvidia-modelopt for INT8 quantization.

In [1]:
# Install packages compatible with Colab's PyTorch 2.7.x (CUDA 12.6 build)
#
# IMPORTANT: Colab ships PyTorch 2.7.x built against CUDA 12.6.
# torchvision must be pinned to the same CUDA build to avoid 'torchvision::nms does not exist'.
# torch-tensorrt must match the PyTorch version exactly to avoid ABI mismatch.

# Pin torchvision to match Colab's PyTorch 2.7.x + cu126 build
# This prevents the 'torchvision::nms does not exist' error that breaks diffusers
!pip install torchvision==0.22.0 --extra-index-url https://download.pytorch.org/whl/cu126 -q

# TensorRT core library
!pip install tensorrt tensorrt-lean -q

# nvidia-modelopt for INT8 PTQ
!pip install 'nvidia-modelopt>=0.15.0' -q

# torch-tensorrt pinned to match Colab's PyTorch 2.7.x
# Must use --extra-index-url to get the version built against PyTorch 2.7
!pip install torch-tensorrt==2.7.0 --extra-index-url https://download.pytorch.org/whl/cu128 -q 2>/dev/null || echo 'torch-tensorrt install failed (will use torch.compile fallback)'

!pip install 'numpy>=1.26.0' -q

# Install t4-diffusion from GitHub
!pip install git+https://github.com/Kash6/t4-diffusion.git -q --no-cache-dir

print("\n✓ Installation complete!")
print("⚠ If this is first install, please restart runtime (Runtime → Restart runtime)")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 136.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.7/897.7 kB 72.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 65.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.0/571.0 MB 78.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 MB 51.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.2/200.2 MB 107.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 183.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.2/158.2 MB 120.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.6/216.6 MB 64.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.8/156.8 MB 111.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.3/201.3 MB 91.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.3/89.3 kB 61.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
# Verify GPU and CUDA environment
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    total_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    cuda_version = torch.version.cuda
    print(f"✓ GPU: {gpu_name}")
    print(f"✓ VRAM: {total_vram:.1f} GB")
    print(f"✓ CUDA: {cuda_version}")
    print(f"✓ PyTorch: {torch.__version__}")

    if "T4" in gpu_name:
        print("✓ Running on T4 GPU (optimal for this package)")
else:
    print("❌ No GPU detected!")
    print("   Go to Runtime → Change runtime type → T4 GPU")

✓ GPU: Tesla T4
✓ VRAM: 15.6 GB
✓ CUDA: 12.6
✓ PyTorch: 2.7.0+cu126
✓ Running on T4 GPU (optimal for this package)


In [3]:
# Verify TensorRT and modelopt installation
trt_available = False
modelopt_available = False

try:
    import tensorrt
    trt_available = True
    print(f"✓ TensorRT: {tensorrt.__version__}")
except ImportError as e:
    print(f"⚠ TensorRT not available: {e}")

try:
    import modelopt
    modelopt_available = True
    # Try to get version
    version = getattr(modelopt, '__version__', 'unknown')
    print(f"✓ nvidia-modelopt: {version}")
except ImportError as e:
    print(f"⚠ nvidia-modelopt not available: {e}")

if trt_available and modelopt_available:
    print("\n🚀 Full TensorRT INT8 optimization available!")
else:
    print("\n📝 Some optimizations may be limited. Check installation above.")

✓ TensorRT: 10.9.0.34
✓ nvidia-modelopt: 0.42.0

🚀 Full TensorRT INT8 optimization available!


## 2. Basic Inference with SDXL-Turbo

In [4]:
from diffusion_trt.pipeline import PipelineConfig, OptimizedPipeline
from diffusion_trt.utils.vram_monitor import VRAMMonitor

# Configure the pipeline
config = PipelineConfig(
    model_id="stabilityai/sdxl-turbo",
    enable_int8=True,           # Enable INT8 quantization
    enable_caching=True,        # Enable feature caching
    num_inference_steps=4,      # SDXL-Turbo uses 4 steps
    guidance_scale=0.0,         # No CFG for SDXL-Turbo
    seed=42,                    # For reproducible results
)

print("Configuration:")
print(f"  Model: {config.model_id}")
print(f"  INT8: {config.enable_int8}")
print(f"  Caching: {config.enable_caching}")
print(f"  Steps: {config.num_inference_steps}")

Configuration:
  Model: stabilityai/sdxl-turbo
  INT8: True
  Caching: True
  Steps: 4


In [5]:
# Load and optimize the model
print("Loading and optimizing model...")
print("This may take a few minutes on first run.")

with VRAMMonitor() as monitor:
    pipeline = OptimizedPipeline.from_pretrained(
        config.model_id,
        config=config,
    )

print(f"\n✓ Model loaded!")
print(f"✓ Peak VRAM: {monitor.peak_gb:.2f} GB")

Loading and optimizing model...
This may take a few minutes on first run.


Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model_index.json:   0%|          | 0.00/685 [00:00<?, ?B/s]

Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

...

Inserted 2382 quantizers


/usr/local/lib/python3.12/dist-packages/modelopt/torch/quantization/model_calib.py:592: UserWarning: time_embedding.linear_1 is not calibrated, skip smoothing
  warnings.warn(f"{name} is not calibrated, skip smoothing")
/usr/local/lib/python3.12/dist-packages/modelopt/torch/quantization/model_calib.py:592: UserWarning: time_embedding.linear_2 is not calibrated, skip smoothing
  warnings.warn(f"{name} is not calibrated, skip smoothing")
/usr/local/lib/python3.12/dist-packages/modelopt/torch/quantization/model_calib.py:592: UserWarning: add_embedding.linear_1 is not calibrated, skip smoothing
  warnings.warn(f"{name} is not calibrated, skip smoothing")
/usr/local/lib/python3.12/dist-packages/modelopt/torch/quantization/model_calib.py:592: UserWarning: add_embedding.linear_2 is not calibrated, skip smoothing
  warnings.warn(f"{name} is not calibrated, skip smoothing")
/usr/local/lib/python3.12/dist-packages/modelopt/torch/quantization/model_calib.py:592: UserWarning: down_blocks.0.resnets

Smoothed 700 modules


/usr/local/lib/python3.12/dist-packages/modelopt/torch/quantization/model_calib.py:592: UserWarning: mid_block.attentions.0.proj_out is not calibrated, skip smoothing
  warnings.warn(f"{name} is not calibrated, skip smoothing")
/usr/local/lib/python3.12/dist-packages/modelopt/torch/quantization/model_calib.py:592: UserWarning: mid_block.resnets.0.time_emb_proj is not calibrated, skip smoothing
  warnings.warn(f"{name} is not calibrated, skip smoothing")
/usr/local/lib/python3.12/dist-packages/modelopt/torch/quantization/model_calib.py:592: UserWarning: mid_block.resnets.1.time_emb_proj is not calibrated, skip smoothing
  warnings.warn(f"{name} is not calibrated, skip smoothing")
ERROR:diffusion_trt.trt_builder:TensorRT compilation failed: argument of type 'NoneType' is not iterable
ERROR:diffusion_trt.trt_builder:TensorRT compilation failed: argument of type 'NoneType' is not iterable



✓ Model loaded!
✓ Peak VRAM: 6.82 GB


In [ ]:
# Generate an image
prompt = "A beautiful sunset over a calm ocean, vibrant colors, photorealistic"

print(f"Generating: '{prompt}'")

with VRAMMonitor() as monitor:
    images = pipeline(prompt)

print(f"\n✓ Generated!")
print(f"✓ VRAM used: {monitor.peak_gb:.2f} GB")

# Display the image
images[0]

Generating: 'A beautiful sunset over a calm ocean, vibrant colors, photorealistic'


  0%|          | 0/4 [00:00<?, ?it/s]

Loading extension modelopt_cuda_ext...


## 3. VRAM Monitoring

In [ ]:
from diffusion_trt.utils.vram_monitor import get_vram_usage, get_peak_vram, clear_cache
from diffusion_trt.models import T4_VRAM_LIMIT_GB

print(f"T4 VRAM Limit: {T4_VRAM_LIMIT_GB} GB")
print(f"Current VRAM: {get_vram_usage():.2f} GB")
print(f"Peak VRAM: {get_peak_vram():.2f} GB")
print(f"Available: {T4_VRAM_LIMIT_GB - get_vram_usage():.2f} GB")

In [ ]:
# Generate multiple images with VRAM monitoring
prompts = [
    "A majestic mountain landscape with snow-capped peaks",
    "A futuristic cityscape at night with neon lights",
    "A cozy cabin in the woods during autumn",
]

for i, prompt in enumerate(prompts, 1):
    with VRAMMonitor() as monitor:
        images = pipeline(prompt)

    print(f"Image {i}: VRAM={monitor.peak_gb:.2f}GB | '{prompt[:40]}...'")
    display(images[0])

## 4. Benchmark

In [ ]:
# Run benchmark
print("Running benchmark (5 iterations + 2 warmup)...")

metrics = pipeline.benchmark(
    prompt="A beautiful landscape with mountains and a lake",
    num_iterations=5,
    warmup_iterations=2,
)

print("\n" + "="*50)
print("BENCHMARK RESULTS")
print("="*50)
print(f"Latency (mean):  {metrics.latency_mean_ms:.0f} ms")
print(f"Latency (std):   {metrics.latency_std_ms:.0f} ms")
print(f"Latency (p50):   {metrics.latency_p50_ms:.0f} ms")
print(f"Latency (p95):   {metrics.latency_p95_ms:.0f} ms")
print(f"Throughput:      {metrics.throughput_images_per_sec:.2f} img/s")
print(f"Peak VRAM:       {metrics.vram_peak_gb:.2f} GB")
print(f"Cache Hit Rate:  {metrics.cache_hit_rate:.1%}")

## 5. Batch Inference

In [ ]:
# Generate multiple images with different prompts
batch_prompts = [
    "A cyberpunk street scene with rain and neon signs",
    "An ancient temple hidden in a misty forest",
    "A steampunk airship flying over Victorian London",
    "A magical library with floating books and glowing orbs",
]

print(f"Generating {len(batch_prompts)} images...")

import time
start = time.time()

batch_images = []
for prompt in batch_prompts:
    images = pipeline(prompt)
    batch_images.extend(images)

total_time = time.time() - start
avg_time = total_time / len(batch_prompts)

print(f"\n✓ Generated {len(batch_images)} images")
print(f"✓ Total time: {total_time:.1f}s")
print(f"✓ Average: {avg_time:.1f}s per image")
print(f"✓ Throughput: {len(batch_images)/total_time:.2f} img/s")

In [ ]:
# Display batch results
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(12, 12))
for ax, img, prompt in zip(axes.flat, batch_images, batch_prompts):
    ax.imshow(img)
    ax.set_title(prompt[:40] + "...", fontsize=10)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 6. Cleanup

In [ ]:
# Clear all caches and free memory
pipeline.clear_cache()
clear_cache()

import gc
gc.collect()
torch.cuda.empty_cache()

print(f"Final VRAM usage: {get_vram_usage():.2f} GB")

---

## Summary

This notebook demonstrated:

1. **Installation** - CUDA 13 compatible TensorRT + nvidia-modelopt packages
2. **Basic Inference** - Generate images with SDXL-Turbo
3. **VRAM Monitoring** - Track memory usage to stay within T4 limits
4. **Benchmarking** - Measure latency and throughput
5. **Batch Inference** - Generate multiple images efficiently

**Expected Performance with INT8 TensorRT on T4:**
- ~0.8-1.0s per image at 512×512 (1.5-2x faster than FP16)
- ~1.0-1.2 images/second throughput
- ~10-12 GB peak VRAM

For more information, visit: https://github.com/Kash6/t4-diffusion